# Generate customer_profile.csv
This notebook generates a synthetic `customer_profile.csv` using Stage 3 rules (10,000 customers by default).

Adjust `N` and `seed` in the code cells before running if you want different totals or reproducible results.

In [15]:
# Install requirements if needed (uncomment to run)
# !pip install numpy pandas faker

In [16]:
import numpy as np
import pandas as pd
from pathlib import Path
from faker import Faker
import random

# Parameters
N = 10000  # total customers
seed = 42  # reproducible results
fake = Faker('en_IN')
Faker.seed(seed)
np.random.seed(seed)
random.seed(seed)

In [17]:
# Employment segments and probabilities (Stage 3 defaults)
segments = ['Salaried','Gig Worker','Self Employed','Student','Business Owner']
probs = [0.35, 0.25, 0.20, 0.10, 0.10]
employment = np.random.choice(segments, size=N, p=probs)

# Basic independent attributes
customer_ids = [f'CUST{str(i+1).zfill(6)}' for i in range(N)]
genders = np.random.choice(['Male','Female','Other'], size=N, p=[0.48,0.48,0.04])

# Age ranges by employment segment
age_ranges = {
    'Salaried': (25, 46),
    'Gig Worker': (22, 39),
    'Self Employed': (28, 51),
    'Student': (18, 27),
    'Business Owner': (30, 56),
}
age = np.empty(N, dtype=int)
for s in segments:
    idx = np.where(employment == s)[0]
    if len(idx) > 0:
        low, high = age_ranges[s]
        age[idx] = np.random.randint(low, high, size=len(idx))

# City tier distributions conditioned on segment
def sample_city_tier(seg, size):
    if seg == 'Salaried':
        return np.random.choice(['Tier 1','Tier 2','Tier 3'], size=size, p=[0.5,0.4,0.1])
    if seg == 'Gig Worker':
        return np.random.choice(['Tier 1','Tier 2','Tier 3'], size=size, p=[0.1,0.5,0.4])
    if seg == 'Student':
        return np.random.choice(['Tier 1','Tier 2','Tier 3'], size=size, p=[0.3,0.4,0.3])
    if seg == 'Self Employed':
        return np.random.choice(['Tier 1','Tier 2','Tier 3'], size=size, p=[0.2,0.5,0.3])
    return np.random.choice(['Tier 1','Tier 2','Tier 3'], size=size, p=[0.2,0.4,0.4])

city_tiers = np.empty(N, dtype=object)
for s in segments:
    idx = np.where(employment == s)[0]
    if len(idx) > 0:
        city_tiers[idx] = sample_city_tier(s, len(idx))

# Income and credit_score based on segment rules
income = np.zeros(N, dtype=int)
credit_score = np.zeros(N, dtype=int)
years_employed = np.zeros(N, dtype=int)
existing_loans = np.zeros(N, dtype=int)

for i, seg in enumerate(employment):
    if seg == 'Salaried':
        income[i] = np.random.randint(40000,150001)
        credit_score[i] = np.random.randint(700,851)
        years_employed[i] = np.random.randint(1,31)
        existing_loans[i] = np.random.poisson(0.3)
    elif seg == 'Gig Worker':
        income[i] = np.random.randint(12000,35001)
        credit_score[i] = np.random.randint(500,681)
        years_employed[i] = np.random.randint(0,20)
        existing_loans[i] = np.random.poisson(0.6)
    elif seg == 'Student':
        # many students have zero income, some small stipends
        income[i] = np.random.choice([0, np.random.randint(1000,20001)], p=[0.7,0.3])
        credit_score[i] = np.random.randint(400,601)
        years_employed[i] = 0
        existing_loans[i] = np.random.poisson(0.2)
    elif seg == 'Self Employed':
        income[i] = np.random.randint(20000,100001)
        credit_score[i] = np.random.randint(550,751)
        years_employed[i] = np.random.randint(1,40)
        existing_loans[i] = np.random.poisson(0.5)
    else:  # Business Owner
        income[i] = np.random.randint(30000,200001)
        credit_score[i] = np.random.randint(600,801)
        years_employed[i] = np.random.randint(2,40)
        existing_loans[i] = np.random.poisson(0.7)

# Acquisition channel (use Stage 3 distribution hints)
acquisition_channel = np.random.choice(['Organic','Social Media','Referral','DSA Agent'], size=N, p=[0.30,0.35,0.20,0.15])

# Education level choices
education_levels = ['School','Graduate','Postgraduate','Professional']
education_level = np.random.choice(education_levels, size=N, p=[0.2,0.5,0.25,0.05])

# Onboarding dates between 2022-01-01 and 2023-12-31
start = pd.to_datetime('2022-01-01')
end = pd.to_datetime('2023-12-31')
onboarding_dates = [fake.date_between(start_date=start, end_date=end).isoformat() for _ in range(N)]

# City names via Faker (en_IN)
city_names = [fake.city() for _ in range(N)]

# Build DataFrame
df = pd.DataFrame({
    'customer_id': customer_ids,
    'age': age,
    'gender': genders,
    'city_tier': city_tiers,
    'city_name': city_names,
    'employment_type': employment,
    'monthly_income': income,
    'credit_score': credit_score,
    'existing_loans': existing_loans,
    'years_employed': years_employed,
    'education_level': education_level,
    'acquisition_channel': acquisition_channel,
    'onboarding_date': onboarding_dates
})

# Make sure types look reasonable
df['monthly_income'] = df['monthly_income'].astype(int)
df['credit_score'] = df['credit_score'].astype(int)
df['existing_loans'] = df['existing_loans'].astype(int)

# Create data directory and save CSV
Path('../data').mkdir(parents=True, exist_ok=True)
out_path = Path('../data/customer_profile.csv')
df.to_csv(out_path, index=False)
print(f'Wrote {len(df)} rows to: {out_path}')
df.head().style

Wrote 10000 rows to: ..\data\customer_profile.csv


,customer_id,age,gender,city_tier,city_name,employment_type,monthly_income,credit_score,existing_loans,years_employed,education_level,acquisition_channel,onboarding_date
0,CUST000001,25,Male,Tier 3,Pudukkottai,Gig Worker,16084,568,1,12,Graduate,Organic,2023-05-12
1,CUST000002,31,Male,Tier 2,Tiruvottiyur,Business Owner,162578,745,0,33,Graduate,Referral,2022-03-28
2,CUST000003,40,Male,Tier 2,Nandyal,Self Employed,87015,560,0,20,Graduate,Organic,2022-01-20
3,CUST000004,34,Female,Tier 1,Anand,Gig Worker,20916,601,0,5,Graduate,Social Media,2023-07-30
4,CUST000005,26,Male,Tier 3,Chapra,Salaried,76553,772,0,23,Postgraduate,DSA Agent,2022-08-02


In [18]:
# Quick validation checks
print('Employment distribution:')
print(df['employment_type'].value_counts(normalize=True).round(3))
print('Acquisition channel distribution:')
print(df['acquisition_channel'].value_counts(normalize=True).round(3))
print('Credit score summary:')
print(df['credit_score'].describe().astype(int))
print('Monthly income summary:')
print(df['monthly_income'].describe().astype(int))

Employment distribution:
employment_type
Salaried          0.356
Gig Worker        0.255
Self Employed     0.193
Student           0.100
Business Owner    0.096
Name: proportion, dtype: float64
Acquisition channel distribution:
acquisition_channel
Social Media    0.344
Organic         0.303
Referral        0.202
DSA Agent       0.151
Name: proportion, dtype: float64
Credit score summary:
count    10000
mean       668
std        105
min        400
25%        586
50%        674
75%        753
max        850
Name: credit_score, dtype: int64
Monthly income summary:
count     10000
mean      63262
std       46431
min           0
25%       24422
50%       54587
75%       95330
max      199945
Name: monthly_income, dtype: int64
